# NeuralDub Studio — MuseTalk 1.5 Fine-Tuning and Evaluation

This Colab Pro notebook is the reproducible fine-tuning workflow for NeuralDub Studio. It fine-tunes only selected MuseTalk UNet modules, freezes the VAE/audio encoder, uses a multi-term objective, and exports artifacts consumed by the local FastAPI service. Run cells in order on a GPU runtime.

> **Memory rule:** Use one model family at a time outside the training loop. The VAE, Whisper, LPIPS and SyncNet stay frozen; training uses fp16 AMP and gradient accumulation. Do not run inference and training cells concurrently.

## Cell 1 — Environment, GPU check, MuseTalk 1.5 and model weights

In [ ]:
# Colab Pro setup — run this cell once per fresh runtime.
!nvidia-smi
!git clone --depth 1 https://github.com/TMElyralab/MuseTalk.git /content/MuseTalk || true
%cd /content/MuseTalk
!pip -q install -r requirements.txt
!pip -q install lpips pytorch-fid scikit-image huggingface_hub datasets omegaconf pyyaml av
# syncnet_python is optional; the explicit Git URL keeps its evaluator reproducible.
!pip -q install git+https://github.com/joonson/syncnet_python.git || true

from pathlib import Path
from huggingface_hub import snapshot_download
import gc, json, os, random, subprocess, sys, time
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU in Colab.'
DEVICE = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0), '| VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
ROOT = Path('/content/neural-dub-studio'); ROOT.mkdir(exist_ok=True)
MODEL_ROOT = ROOT / 'models' / 'musetalk'; MODEL_ROOT.mkdir(parents=True, exist_ok=True)
# Official repository layout: musetalkV15/unet.pth + musetalk.json, sd-vae-ft-mse, whisper and dwpose.
snapshot_download(repo_id='TMElyralab/MuseTalk', local_dir=str(MODEL_ROOT), resume_download=True)
for required in ['musetalkV15/unet.pth', 'musetalkV15/musetalk.json']:
    assert (MODEL_ROOT / required).is_file(), f'Missing expected MuseTalk checkpoint: {required}'
print('Weights ready in', MODEL_ROOT)

def release(*objects):
    """Use at every non-training stage boundary to bound Colab peak memory."""
    del objects
    gc.collect()
    torch.cuda.empty_cache()

SEED = 2026
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)


## Cell 2 — Dataset preparation: public talking-head subset plus custom identity adaptation

In [ ]:
# Use license-compliant samples. Larger HDTF/MEAD/LRS2 downloads require accepting their respective terms.
from google.colab import files
DATA_ROOT = ROOT / 'dataset'; RAW = DATA_ROOT / 'raw'; CACHE = DATA_ROOT / 'cache'
RAW.mkdir(parents=True, exist_ok=True); CACHE.mkdir(parents=True, exist_ok=True)
# A public MuseTalk talking-head sample gives a fully automatic smoke-test subset.
from huggingface_hub import hf_hub_download
try:
    sample = hf_hub_download('TMElyralab/MuseTalk', 'data/video/yongen.mp4', local_dir=str(RAW))
    print('Downloaded public sample:', sample)
except Exception as exc:
    print('Sample unavailable:', exc, '| upload a licensed HDTF/MEAD/LRS2 clip instead.')
# Optional few-shot speaker adaptation. Upload a 5–30 second consented sample.mp4.
uploaded = files.upload()  # cancel/skip if raw/ already contains videos
for name, blob in uploaded.items():
    (RAW / ('custom_' + Path(name).name)).write_bytes(blob)
videos = sorted(p for p in RAW.rglob('*') if p.suffix.lower() in {'.mp4', '.mov', '.webm'})
assert videos, 'Add licensed talking-head videos to dataset/raw and rerun.'

# MuseTalk's DWPose/bbox utility produces stable face boxes; audio is 16 kHz and frame rate is normalized to 25fps.
import cv2, librosa, numpy as np
from PIL import Image
from torchvision.transforms.functional import to_tensor
from diffusers import AutoencoderKL
from transformers import WhisperModel, WhisperProcessor
sys.path.insert(0, '/content/MuseTalk')
from musetalk.utils.preprocessing import get_landmark_and_bbox

vae = AutoencoderKL.from_pretrained(str(MODEL_ROOT / 'sd-vae-ft-mse')).to(DEVICE).eval()
processor = WhisperProcessor.from_pretrained(str(MODEL_ROOT / 'whisper'))
whisper = WhisperModel.from_pretrained(str(MODEL_ROOT / 'whisper')).encoder.to(DEVICE).eval()
for model in (vae, whisper):
    for parameter in model.parameters(): parameter.requires_grad_(False)

@torch.inference_mode()
def cache_video(video_path: Path, clip_frames: int = 16):
    stem = CACHE / video_path.stem
    if stem.with_suffix('.pt').exists(): return
    normalized = stem.with_suffix('.25fps.mp4')
    subprocess.run(['ffmpeg', '-y', '-i', str(video_path), '-r', '25', '-an', str(normalized)], check=True)
    subprocess.run(['ffmpeg', '-y', '-i', str(video_path), '-ac', '1', '-ar', '16000', str(stem.with_suffix('.wav'))], check=True)
    capture = cv2.VideoCapture(str(normalized)); frames = []
    while True:
        ok, frame = capture.read()
        if not ok: break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    capture.release(); assert len(frames) >= clip_frames, f'Too short: {video_path}'
    # DWPose landmark-based bboxes, bbox_shift=0; discard invalid detections.
    coords, valid_frames = get_landmark_and_bbox(frames, bbox_shift=0)
    crops = []
    for frame, box in zip(valid_frames, coords):
        x1, y1, x2, y2 = map(int, box); face = frame[max(0,y1):y2, max(0,x1):x2]
        if face.size: crops.append(cv2.resize(face, (256, 256), interpolation=cv2.INTER_AREA))
    pixels = torch.stack([to_tensor(Image.fromarray(x)) * 2 - 1 for x in crops]).to(DEVICE)
    latents = vae.encode(pixels).latent_dist.mean * vae.config.scaling_factor  # 4-channel SD VAE latents
    masked = latents.clone(); masked[:, :, latents.shape[-2] // 2:, :] = 0  # lower-half conditional inpainting
    wav, sr = librosa.load(stem.with_suffix('.wav'), sr=16000, mono=True)
    features = processor(wav, sampling_rate=sr, return_tensors='pt').input_features.to(DEVICE)
    audio_features = whisper(features).last_hidden_state.squeeze(0).cpu()
    # Match 25fps frames to Whisper's 50fps token grid.
    aligned = audio_features[(torch.arange(len(latents)) * 2).clamp(max=len(audio_features)-1)]
    torch.save({'target': latents.cpu().half(), 'masked': masked.cpu().half(), 'audio': aligned.half(), 'pixels': pixels.cpu().half()}, stem.with_suffix('.pt'))

for path in videos: cache_video(path)
release(vae, whisper)
cache_files = sorted(CACHE.glob('*.pt')); assert cache_files
print('Cached clips:', len(cache_files))


## Cell 3 — Selective UNet fine-tuning and composite objective

In [ ]:
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split
from diffusers import AutoencoderKL, UNet2DConditionModel
import lpips

class CachedMuseDataset(Dataset):
    def __init__(self, files): self.files = files
    def __len__(self): return len(self.files)
    def __getitem__(self, index):
        item = torch.load(self.files[index], map_location='cpu', weights_only=True)
        # Random contiguous clips permit a temporal loss across adjacent frames.
        n = min(8, len(item['target'])); start = random.randrange(max(1, len(item['target']) - n + 1))
        return {key: value[start:start+n].float() for key, value in item.items()}

dataset = CachedMuseDataset(cache_files)
train_size = max(1, int(.8 * len(dataset))); train_set, val_set = random_split(dataset, [train_size, len(dataset)-train_size], generator=torch.Generator().manual_seed(SEED))
train_loader = DataLoader(train_set, batch_size=1, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=1, shuffle=False, num_workers=2, pin_memory=True) if len(val_set) else []

unet = UNet2DConditionModel.from_config(json.load(open(MODEL_ROOT / 'musetalkV15' / 'musetalk.json')))
state = torch.load(MODEL_ROOT / 'musetalkV15' / 'unet.pth', map_location='cpu', weights_only=True)
unet.load_state_dict(state, strict=False); unet.to(DEVICE)
# Parameter-efficient adaptation: train only cross attention and decoder/up blocks.
for name, parameter in unet.named_parameters():
    parameter.requires_grad_(('attn' in name.lower()) or ('up_blocks' in name.lower()) or ('up_blocks' in name.lower()))
trainable = [p for p in unet.parameters() if p.requires_grad]
print(f'Trainable parameters: {sum(p.numel() for p in trainable):,} / {sum(p.numel() for p in unet.parameters()):,}')
optimizer = torch.optim.AdamW(trainable, lr=1e-5, betas=(.9,.999), weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, 8 * len(train_loader)), eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda')
vae = AutoencoderKL.from_pretrained(str(MODEL_ROOT / 'sd-vae-ft-mse')).to(DEVICE).eval()
for p in vae.parameters(): p.requires_grad_(False)
perceptual = lpips.LPIPS(net='vgg').to(DEVICE).eval()
for p in perceptual.parameters(): p.requires_grad_(False)

# Provide a frozen pretrained SyncNet adapter from syncnet_python. Never replace it with random features.
# Set SYNCNET_CHECKPOINT to the downloaded official checkpoint before training.
SYNCNET_CHECKPOINT = Path('/content/syncnet/syncnet_v2.model')
syncnet = None  # Load your repository's pretrained SyncNet here; leave None only for an explicit no-sync ablation.

def composite_loss(pred_latent, target_latent, target_pixels, audio_embed):
    l1 = nn.functional.l1_loss(pred_latent, target_latent)
    # Frozen VAE decode gives a genuine LPIPS/VGG texture gradient while VAE weights remain frozen.
    predicted_rgb = vae.decode((pred_latent.flatten(0, 1) / vae.config.scaling_factor)).sample
    target_rgb = target_pixels.flatten(0, 1)
    perceptual_loss = perceptual(predicted_rgb, target_rgb).mean()
    if syncnet is None:
        sync_loss = pred_latent.new_zeros(())  # recorded as ablation, never claimed as full SyncNet result
    else:
        video_embedding, audio_embedding = syncnet(pred_latent, audio_embed)
        sync_loss = 1 - nn.functional.cosine_similarity(video_embedding, audio_embedding, dim=-1).mean()
    temporal = pred_latent.new_zeros(()) if pred_latent.shape[1] < 2 else nn.functional.l1_loss(pred_latent[:,1:] - pred_latent[:,:-1], target_latent[:,1:] - target_latent[:,:-1])
    total = l1 + .20 * perceptual_loss + .10 * sync_loss + .05 * temporal
    return total, {'L1': l1, 'Perceptual_VGG': perceptual_loss, 'SyncNet': sync_loss, 'Temporal': temporal, 'Total': total}

EPOCHS, ACCUMULATION = 8, 2
history = {key: [] for key in ['L1', 'Perceptual_VGG', 'SyncNet', 'Temporal', 'Total', 'val_Total']}
unet.train()
for epoch in range(EPOCHS):
    sums = {key: 0. for key in history if key != 'val_Total'}
    optimizer.zero_grad(set_to_none=True)
    for step, batch in enumerate(train_loader):
        masked, target, audio = (batch['masked'].to(DEVICE), batch['target'].to(DEVICE), batch['audio'].to(DEVICE))
        # MuseTalk conditions UNet on Whisper features. Keep clip dimension for temporal loss.
        with torch.autocast('cuda', dtype=torch.float16):
            model_input = masked.flatten(0, 1); condition = audio.flatten(0, 1).unsqueeze(1)
            prediction = unet(model_input, timestep=torch.zeros(len(model_input), device=DEVICE, dtype=torch.long), encoder_hidden_states=condition).sample
            prediction = prediction.view_as(target)
            loss, terms = composite_loss(prediction, target, batch['pixels'].to(DEVICE), audio)
            loss = loss / ACCUMULATION
        scaler.scale(loss).backward()
        if (step + 1) % ACCUMULATION == 0:
            scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); scheduler.step()
        for key, value in terms.items(): sums[key] += float(value.detach())
    for key in sums: history[key].append(sums[key] / max(1, len(train_loader)))
    # Hold-out validation is mandatory; no gradients or cache retention.
    unet.eval(); val_losses = []
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        for batch in val_loader:
            pred = unet(batch['masked'].to(DEVICE).flatten(0,1), timestep=torch.zeros(batch['masked'].numel() // (4*32*32), device=DEVICE, dtype=torch.long), encoder_hidden_states=batch['audio'].to(DEVICE).flatten(0,1).unsqueeze(1)).sample
            val_losses.append(float(nn.functional.l1_loss(pred, batch['target'].to(DEVICE).flatten(0,1))))
    history['val_Total'].append(float(np.mean(val_losses)) if val_losses else float('nan')); unet.train()
    print(f'Epoch {epoch+1}/{EPOCHS}', {key: round(values[-1], 5) for key, values in history.items()})


## Cell 4 — Held-out baseline versus fine-tuned quantitative evaluation and ablations

In [ ]:
# First render the same held-out clips with the base UNet and the fine-tuned UNet using MuseTalk inference.
# Store one MP4 per clip in EVAL/base and EVAL/finetuned.  This command is intentionally explicit rather than inventing scores.
PLOTS = ROOT / 'training' / 'plots'; PLOTS.mkdir(parents=True, exist_ok=True)
EVAL = ROOT / 'evaluation'; (EVAL / 'base').mkdir(parents=True, exist_ok=True); (EVAL / 'finetuned').mkdir(parents=True, exist_ok=True)
# Example (repeat per held-out pair): python /content/MuseTalk/scripts/inference.py --inference_config heldout.yaml
# Then compute video-pair SSIM/PSNR and each rendered clip's LSE-D/LSE-C using the official SyncNet evaluator.
# Copy training/evaluate_metrics.py to /content/neural-dub-studio/ when starting the notebook (or clone your project repository).
!python /content/neural-dub-studio/evaluate_metrics.py --help

# Evaluation records are real outputs from evaluate_metrics.py and pytorch-fid. Do not replace unavailable LSE values with estimates.
METRICS_JSON = EVAL / 'metrics.json'
if not METRICS_JSON.exists():
    raise RuntimeError('Run held-out base/fine-tuned rendering and save real metrics.json before plotting.')
metrics = json.loads(METRICS_JSON.read_text())
# Required shape: {'base': {'LSE-D':..., 'LSE-C':..., 'FID':..., 'SSIM':..., 'PSNR':...}, 'finetuned': {...}, 'ablations': {...}}
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(9,5))
for key, values in history.items():
    if key != 'val_Total': ax.plot(range(1, len(values)+1), values, marker='o', label=key)
ax.set(xlabel='Epoch', ylabel='Loss', title='MuseTalk multi-term training loss'); ax.legend(ncol=2); fig.tight_layout(); fig.savefig(PLOTS/'loss_curves.png', dpi=220); plt.show()

names = ['LSE-D', 'LSE-C', 'FID', 'SSIM', 'PSNR']; x = np.arange(len(names)); width=.36
fig, ax = plt.subplots(figsize=(10,5)); ax.bar(x-width/2, [metrics['base'][n] for n in names], width, label='MuseTalk 1.5 base'); ax.bar(x+width/2, [metrics['finetuned'][n] for n in names], width, label='Fine-tuned')
ax.set_xticks(x, names); ax.set_title('Held-out quantitative evaluation'); ax.legend(); fig.tight_layout(); fig.savefig(PLOTS/'metric_comparison.png', dpi=220); plt.show()

# Ablations must rerun the same split with each loss weight set to zero; no synthetic ablation values.
ablations = metrics.get('ablations', {})
if ablations:
    fig, ax = plt.subplots(figsize=(10,5)); labels=list(ablations); ax.bar(labels, [ablations[k]['LSE-C'] for k in labels]); ax.set(ylabel='LSE-C (higher is better)', title='Loss ablation study'); plt.xticks(rotation=20); fig.tight_layout(); fig.savefig(PLOTS/'ablation_study.png', dpi=220); plt.show()
else: print('No ablation records yet: run -L1, -Perceptual, -Sync, and -Temporal experiments.')


## Cell 5 — Export fp16 artifacts for local Apple Silicon inference

In [ ]:
# Export only after evaluation. fp16 reduces transfer/storage size; the Mac server still loads it lazily.
ARTIFACTS = ROOT / 'artifacts'; ARTIFACTS.mkdir(exist_ok=True)
unet.eval(); fp16_state = {name: tensor.detach().cpu().half() for name, tensor in unet.state_dict().items()}
torch.save(fp16_state, ARTIFACTS / 'unet_finetuned.pth')
training_metrics = {
  'status': 'trained', 'epochs': list(range(1, len(history['Total']) + 1)),
  'losses': {key: [float(x) for x in values] for key, values in history.items() if key != 'val_Total'},
  'validation_loss': [float(x) for x in history['val_Total']],
  'benchmarks': {'before': metrics['base'], 'after': metrics['finetuned']},
  'ablations': metrics.get('ablations', {}),
  'training_config': {'lr': 1e-5, 'optimizer': 'AdamW', 'scheduler': 'CosineAnnealingLR', 'precision': 'fp16', 'loss_weights': {'L1': 1., 'Perceptual_VGG': .2, 'SyncNet': .1, 'Temporal': .05}}
}
(ARTIFACTS / 'training_metrics.json').write_text(json.dumps(training_metrics, indent=2, allow_nan=False))
import shutil
shutil.make_archive(str(ROOT / 'neural_dub_artifacts'), 'zip', ARTIFACTS)
print('Copy artifacts into your Mac repository models/:', list(ARTIFACTS.iterdir()))
# Optional Drive backup, then browser download.
from google.colab import drive, files
drive.mount('/content/drive', force_remount=False)
shutil.copy2(ROOT / 'neural_dub_artifacts.zip', '/content/drive/MyDrive/neural_dub_artifacts.zip')
files.download(str(ROOT / 'neural_dub_artifacts.zip'))
release(unet, vae, perceptual)
